# สร้างคุณลักษณะ NSL-KDD (Feature Engineering)

Notebook นี้เปลี่ยนข้อมูลที่ผ่าน cleansing ให้อยู่ในรูปแบบตัวเลขที่โมเดลนำไปใช้ได้ และเพิ่มตัวแทนคุณลักษณะบางแบบที่อาจช่วยให้ตรวจรูปแบบ traffic ได้ง่ายขึ้น งานหลักคือ log transform สำหรับค่าหางยาว, indicator สำหรับเหตุการณ์หายาก, aggregate สำหรับสรุป bytes/error และ One-Hot Encoding สำหรับ protocol/service/flag

สิ่งสำคัญคือแยก `features` ออกจาก `targets` (`label`, `attack_family`, `is_attack`) คนละไฟล์เพื่อป้องกัน target leakage; vocabulary ของ One-Hot เรียนจาก KDDTrain เท่านั้น แล้วบังคับให้ test ใช้ลำดับคอลัมน์เดียวกัน ข้อมูลไม่ถูก scale ที่นี่ เพราะ scaler ต้อง fit เฉพาะ train ภายใน pipeline ของแต่ละโมเดล

ผลที่คาดจากชุดปัจจุบัน: 37 ตัวเลขต้นฉบับ + 36 คุณลักษณะสร้างเพิ่ม + 84 One-Hot = 157 feature columns และแยก 3 target columns ออกต่างหาก ทั้งสาม split ควรมีจำนวนแถวเดิมและ schema เดียวกัน

ข้อควรทราบสำหรับโปรเจกต์: experiment Isolation Forest ปัจจุบันตั้งใจใช้ baseline 121 คอลัมน์ (37 ตัวเลขต้นฉบับ + 84 One-Hot) และยังไม่ใช้ 36 engineered alternatives เพื่อให้เปรียบเทียบ baseline ได้อย่างเป็นธรรม ผล engineered ทั้งหมดจึงถูกส่งออกไว้สำหรับทดลอง/ทำ ablation ในโมเดลถัดไป ไม่ได้หมายความว่าทุกโมเดลต้องใช้ทุกคอลัมน์

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. โหลดไฟล์ที่ผ่าน cleansing และตรวจภาพรวม

ค้นหา project root จากโฟลเดอร์ที่เปิด notebook แล้วอ่าน CSV สะอาดของ KDDTrain, KDDTest และ KDDTest-21 จาก `output/NSL-KDD/clean/` ทำข้อความในคอลัมน์หมวดหมู่/label ให้เป็นมาตรฐานซ้ำอีกครั้งก่อนสร้าง encoding

ตารางแสดงจำนวนแถว คอลัมน์ และ missing cells ของแต่ละ split ปกติควรเห็น 125,973 / 22,544 / 11,850 แถว และ missing เป็นศูนย์ หากไม่พบไฟล์ โค้ดจะบอกให้รัน cleansing ก่อน; ถ้าจำนวนแถวหรือ missing ต่างจากที่คาดควรตรวจ pipeline ก่อนสร้าง feature

In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "download_nsl_dataset.py").is_file() and (candidate / "data-pipeline").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the CSI_ML_Final project root from the current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
PIPELINE_OUTPUT_DIR = PROJECT_ROOT / "data-pipeline" / "output" / "NSL-KDD"
CLEAN_DIR = PIPELINE_OUTPUT_DIR / "clean"
OUTPUT_DIR = PIPELINE_OUTPUT_DIR / "feature_engineered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILES = {
    "KDDTrain": CLEAN_DIR / "KDDTrain_clean.csv",
    "KDDTest": CLEAN_DIR / "KDDTest_clean.csv",
    "KDDTest-21": CLEAN_DIR / "KDDTest-21_clean.csv",
}

for dataset_name, path in DATA_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"ไม่พบ {dataset_name}: {path.resolve()} "
            "กรุณารัน nsl_cleansing.ipynb ก่อน"
        )

datasets = {name: pd.read_csv(path) for name, path in DATA_FILES.items()}

categorical_cols = ["protocol_type", "service", "flag"]
target_cols = ["label", "attack_family", "is_attack"]

for frame in datasets.values():
    for col in categorical_cols + ["label", "attack_family"]:
        frame[col] = frame[col].astype("string").str.strip().str.lower()

summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in datasets.items()
])
display(summary)

,dataset,rows,columns,missing_cells
0,KDDTrain,125973,43,0
1,KDDTest,22544,43,0
2,KDDTest-21,11850,43,0


## 2. ตรวจ schema และแยก input ออกจากคำตอบ

ตรวจว่าทั้งสาม split มีชื่อคอลัมน์ตรงกับ KDDTrain และยืนยันว่า `difficulty` กับ `num_outbound_cmds` ถูก cleansing ตัดไปแล้ว จากนั้นแยก `label`, `attack_family`, `is_attack` เป็น target; คอลัมน์ที่เหลือแบ่งเป็น categorical 3 ตัว (`protocol_type`, `service`, `flag`) และ numeric inputs

ผลควรแสดง categorical 3 คอลัมน์, numeric input 37 คอลัมน์ และ target 3 คอลัมน์ การแยกนี้ช่วยป้องกันโมเดลเห็นคำตอบล่วงหน้า โดยเฉพาะ `is_attack` ซึ่งบอกตรง ๆ ว่าเป็น attack หรือไม่ หากพบคอลัมน์ต้องห้ามหรือ schema ไม่ตรง โค้ดจะหยุดแทนการสร้างชุดข้อมูลที่ใช้เทียบกันไม่ได้

In [3]:
train_columns = set(datasets["KDDTrain"].columns)
for dataset_name, frame in datasets.items():
    if set(frame.columns) != train_columns:
        raise ValueError(f"schema ของ {dataset_name} ไม่ตรงกับ KDDTrain")

for forbidden_col in ["difficulty", "num_outbound_cmds"]:
    if forbidden_col in train_columns:
        raise ValueError(
            f"ยังพบ {forbidden_col}; กรุณาตรวจ output จาก cleansing ก่อน"
        )

missing_targets = set(target_cols) - train_columns
if missing_targets:
    raise ValueError(f"ไม่พบ target columns: {sorted(missing_targets)}")

numeric_cols = [
    col for col in datasets["KDDTrain"].columns
    if col not in categorical_cols + target_cols
    and pd.api.types.is_numeric_dtype(datasets["KDDTrain"][col])
]

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric input columns: {len(numeric_cols)}")
print(f"Target columns: {target_cols}")

Categorical columns: ['protocol_type', 'service', 'flag']
Numeric input columns: 37
Target columns: ['label', 'attack_family', 'is_attack']


## 3. เพิ่ม log1p ให้ตัวแปรที่มีหางยาว

เลือกตัวเลขไม่ติดลบที่มักกระจายเบ้ เช่น duration, bytes, connection counts และ security counts แล้วสร้าง `log1p_x = ln(1+x)` เพิ่มอีกหนึ่งคอลัมน์ต่อหนึ่งตัวแปร ค่าเดิมยังคงอยู่เพื่อให้โมเดล/การทดลองเลือกใช้ได้ ไม่ได้แทนที่ข้อมูลต้นฉบับ

มี 13 คอลัมน์ที่ถูกแปลง ตัวอย่าง 0, 10 และ 1,000,000 จะกลายเป็นประมาณ 0, 2.40 และ 13.82 จึงลดระยะห่างของค่าสูงมาก แต่ยังรักษาลำดับและค่าศูนย์ไว้ ผลนี้มีประโยชน์กับโมเดลที่ไวต่อสเกล/ระยะห่าง; สำหรับ Isolation Forest ผลของ monotonic transform อาจต่างจากโมเดลเชิงระยะทาง จึงควรทดลองแยกเป็น ablation แทนการสมมติว่าจะดีขึ้นเสมอ

In [4]:
log_cols = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "count",
    "srv_count",
    "dst_host_count",
    "dst_host_srv_count",
    "hot",
    "num_failed_logins",
    "num_compromised",
    "num_root",
    "num_file_creations",
    "num_access_files",
]
log_cols = [col for col in log_cols if col in numeric_cols]

log_feature_names = ["log1p_" + col for col in log_cols]
print(f"สร้าง log1p features {len(log_feature_names)} ตัว:")
print(log_feature_names)

สร้าง log1p features 13 ตัว:
['log1p_duration', 'log1p_src_bytes', 'log1p_dst_bytes', 'log1p_count', 'log1p_srv_count', 'log1p_dst_host_count', 'log1p_dst_host_srv_count', 'log1p_hot', 'log1p_num_failed_logins', 'log1p_num_compromised', 'log1p_num_root', 'log1p_num_file_creations', 'log1p_num_access_files']


## 4. สร้าง indicator สำหรับเหตุการณ์ที่เกิดไม่บ่อย

สำหรับ 12 ตัวแปร เช่น land, wrong_fragment, urgent, failed login, root/file/shell activity จะสร้าง `has_x = 1` เมื่อค่าต้นฉบับมากกว่า 0 และเป็น 0 เมื่อไม่เกิดเหตุการณ์ นอกจากนี้สร้างกลุ่มรวมอีก 2 ตัว: `has_root_activity` และ `has_file_or_shell_activity`

จึงเพิ่ม indicator ทั้งหมด 14 คอลัมน์ ค่า 0/1 ทำให้สื่อสารว่าเหตุการณ์เกิดหรือไม่ชัดเจน แม้ค่าต้นฉบับมีศูนย์เกือบทั้งหมด เช่น `num_failed_logins=0` จะให้ `has_num_failed_logins=0`; หากมีตั้งแต่หนึ่งครั้งจะเป็น 1 การสร้าง indicator ไม่ได้เพิ่มข้อมูลใหม่ แต่ช่วยเปลี่ยนรูปแบบแทนข้อมูลให้เห็นสัญญาณ rare event ชัดขึ้น

In [5]:
indicator_cols = [
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
]
indicator_cols = [col for col in indicator_cols if col in numeric_cols]

indicator_feature_names = ["has_" + col for col in indicator_cols]
group_feature_names = ["has_root_activity", "has_file_or_shell_activity"]
print(f"สร้าง indicator features {len(indicator_feature_names) + len(group_feature_names)} ตัว")

สร้าง indicator features 14 ตัว


## 5. สร้างตัวแปรสรุป traffic และ error

ฟังก์ชันสร้างคุณลักษณะรวม 9 ตัว ได้แก่ `total_bytes` และ log ของมัน, ผลต่าง bytes แบบ signed-log, ค่าเฉลี่ย SYN error / reject error / overall error, อัตราส่วนจำนวน service ต่อ host, ช่องว่างระหว่าง service rate กับ host service rate และค่าเฉลี่ย error ระดับ host ตัวแปรเหล่านี้สรุปข้อมูลหลายคอลัมน์ให้เป็นสัญญาณที่มีความหมายเชิงปริมาณมากขึ้น

ตัวอย่าง `total_bytes = src_bytes + dst_bytes` คือปริมาณข้อมูลรวมสองทิศทาง; `signed_log_byte_difference` เป็นบวกเมื่อ source ส่งมากกว่า destination และเป็นลบเมื่อ destination ส่งมากกว่า โดย log ลดอิทธิพลของค่ามหาศาล; `host_service_rate_gap` บอกความต่างของความสม่ำเสมอของ service ในบริบท host เทียบ connection

หลีกเลี่ยง `count - srv_count` หรือ `srv_count / count` เพราะสองตัวนับคนละนิยาม/กลุ่มและอาจทำให้ตีความเป็นจำนวนหรือสัดส่วนที่ไม่มีความหมาย ส่วน ratio ที่สร้างใช้ `dst_host_count + 1` เพื่อกันหารศูนย์ แต่ควรตรวจ distribution ของค่านี้ก่อนนำไปใช้จริง คุณลักษณะที่สร้างเป็นสมมติฐานที่ต้องทดสอบ ไม่ใช่การรับประกันว่าโมเดลดีขึ้น

In [6]:
engineered_descriptions = {
    "total_bytes": "src_bytes + dst_bytes: ปริมาณข้อมูลรวมสองทิศทาง",
    "log1p_total_bytes": "log1p(total_bytes): ปริมาณรวมในสเกลที่บีบช่วงแล้ว",
    "signed_log_byte_difference": "sign(src-dst) * log1p(abs(src-dst)): ทิศทางและขนาดความต่างของ bytes",
    "syn_error_mean": "ค่าเฉลี่ย serror_rate และ srv_serror_rate",
    "reject_error_mean": "ค่าเฉลี่ย rerror_rate และ srv_rerror_rate",
    "overall_error_rate": "ค่าเฉลี่ย error rate ระดับ connection/service",
    "host_service_count_ratio": "dst_host_srv_count / (dst_host_count + 1)",
    "host_service_rate_gap": "dst_host_same_srv_rate - same_srv_rate",
    "host_error_mean": "ค่าเฉลี่ย error rate สี่ตัวในระดับ destination host",
    "has_root_activity": "มี root_shell, su_attempted หรือ num_root มากกว่า 0",
    "has_file_or_shell_activity": "มี file creation, shell หรือ access file มากกว่า 0",
}

def build_engineered_columns(frame):
    result = pd.DataFrame(index=frame.index)

    for col in log_cols:
        result["log1p_" + col] = np.log1p(frame[col].clip(lower=0))

    for col in indicator_cols:
        result["has_" + col] = (frame[col] > 0).astype("int8")

    result["has_root_activity"] = (
        (frame["root_shell"] > 0)
        | (frame["su_attempted"] > 0)
        | (frame["num_root"] > 0)
    ).astype("int8")

    result["has_file_or_shell_activity"] = (
        (frame["num_file_creations"] > 0)
        | (frame["num_shells"] > 0)
        | (frame["num_access_files"] > 0)
    ).astype("int8")

    result["total_bytes"] = frame["src_bytes"] + frame["dst_bytes"]
    result["log1p_total_bytes"] = np.log1p(result["total_bytes"].clip(lower=0))

    byte_diff = frame["src_bytes"] - frame["dst_bytes"]
    result["signed_log_byte_difference"] = (
        np.sign(byte_diff) * np.log1p(byte_diff.abs())
    )
    result["syn_error_mean"] = (
        frame["serror_rate"] + frame["srv_serror_rate"]
    ) / 2
    result["reject_error_mean"] = (
        frame["rerror_rate"] + frame["srv_rerror_rate"]
    ) / 2
    result["overall_error_rate"] = frame[
        ["serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate"]
    ].mean(axis=1)

    result["host_service_count_ratio"] = (
        frame["dst_host_srv_count"] / (frame["dst_host_count"] + 1)
    )
    result["host_service_rate_gap"] = (
        frame["dst_host_same_srv_rate"] - frame["same_srv_rate"]
    )
    host_error_cols = [
        "dst_host_serror_rate",
        "dst_host_srv_serror_rate",
        "dst_host_rerror_rate",
        "dst_host_srv_rerror_rate",
    ]
    result["host_error_mean"] = frame[host_error_cols].mean(axis=1)

    return result

engineered_preview = build_engineered_columns(datasets["KDDTrain"])
display(engineered_preview.head())
print(f"เพิ่ม engineered numeric features: {engineered_preview.shape[1]} ตัว")

,log1p_duration,log1p_src_bytes,log1p_dst_bytes,log1p_count,log1p_srv_count,log1p_dst_host_count,log1p_dst_host_srv_count,log1p_hot,log1p_num_failed_logins,log1p_num_compromised,log1p_num_root,log1p_num_file_creations,log1p_num_access_files,has_land,has_wrong_fragment,has_urgent,has_hot,has_num_failed_logins,has_num_compromised,has_root_shell,has_su_attempted,has_num_root,has_num_file_creations,has_num_shells,has_num_access_files,has_root_activity,has_file_or_shell_activity,total_bytes,log1p_total_bytes,signed_log_byte_difference,syn_error_mean,reject_error_mean,overall_error_rate,host_service_count_ratio,host_service_rate_gap,host_error_mean
0,0.0000,6.1985,0.0000,1.0986,1.0986,5.0173,3.2581,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,491,6.1985,6.1985,0.0000,0.0000,0.0000,0.1656,-0.8300,0.0125
1,0.0000,4.9904,0.0000,2.6391,0.6931,5.5452,0.6931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,146,4.9904,4.9904,0.0000,0.0000,0.0000,0.0039,-0.0800,0.0000
2,0.0000,0.0000,0.0000,4.8203,1.9459,5.5452,3.2958,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,1.0000,0.0000,0.5000,0.1016,0.0500,0.5000
3,0.0000,5.4510,9.0063,1.7918,1.7918,3.4340,5.5452,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8385,9.0343,-8.9774,0.2000,0.0000,0.1000,8.2258,0.0000,0.0125
4,0.0000,5.2983,6.0426,3.4340,3.4965,5.5452,5.5452,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,619,6.4297,-5.4027,0.0000,0.0000,0.0000,0.9961,0.0000,0.0000


เพิ่ม engineered numeric features: 36 ตัว


## 6. แปลงหมวดหมู่เป็น One-Hot columns

ข้อความ `protocol_type`, `service`, `flag` ยังนำเข้าโมเดลตัวเลขโดยตรงไม่ได้ จึงสร้างคอลัมน์ 0/1 เช่น `protocol_type=tcp` มีค่า 1 เมื่อแถวนั้นใช้ TCP และ 0 เมื่อไม่ใช่ โค้ดเรียนรู้หมวดหมู่และลำดับคอลัมน์จาก KDDTrain เท่านั้น เพื่อป้องกัน test แอบกำหนด schema ให้ตัวเอง

ชุดนี้มี 3 protocol + 70 service + 11 flag รวม 84 One-Hot columns รายงาน unknown category ตรวจหมวดในแต่ละ split ที่ไม่มีใน train; หากพบ หมวดนั้นจะไม่มีคอลัมน์ตรงกันและกลุ่ม one-hot จะเป็นศูนย์ พร้อมบันทึกไว้ให้ตรวจ ไม่ควร fit encoder แยกบน test เพราะจะทำให้ความหมาย/ลำดับ feature ระหว่าง train กับ test ไม่ตรงกัน

In [7]:
train = datasets["KDDTrain"]
train_categories = {
    col: sorted(train[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols
}

def encode_categories(frame):
    categorical = frame[categorical_cols].copy()
    for col in categorical_cols:
        categorical[col] = categorical[col].astype("string").fillna("__missing__")

    return pd.get_dummies(
        categorical,
        prefix=categorical_cols,
        prefix_sep="=",
        dtype="int8",
    )

train_ohe = encode_categories(train)
ohe_columns = train_ohe.columns.tolist()

unknown_category_rows = []
for dataset_name, frame in datasets.items():
    for col in categorical_cols:
        unseen = set(frame[col].dropna().astype(str).unique()) - set(train_categories[col])
        rows_with_unseen = int(frame[col].astype(str).isin(unseen).sum())
        unknown_category_rows.append({
            "dataset": dataset_name,
            "column": col,
            "unseen_categories": ", ".join(sorted(unseen)),
            "unseen_category_count": len(unseen),
            "rows_with_unseen_category": rows_with_unseen,
        })

unknown_category_report = pd.DataFrame(unknown_category_rows)
display(unknown_category_report)

,dataset,column,unseen_categories,unseen_category_count,rows_with_unseen_category
0,KDDTrain,protocol_type,,0,0
1,KDDTrain,service,,0,0
2,KDDTrain,flag,,0,0
3,KDDTest,protocol_type,,0,0
4,KDDTest,service,,0,0
5,KDDTest,flag,,0,0
6,KDDTest-21,protocol_type,,0,0
7,KDDTest-21,service,,0,0
8,KDDTest-21,flag,,0,0


## 7. รวม feature matrix และเก็บ target แยกไฟล์

แต่ละ split ถูกประกอบจากตัวเลขต้นฉบับ 37 ตัว, engineered features 36 ตัว (13 log + 14 indicator + 9 aggregate) และ One-Hot 84 ตัว รวมเป็น 157 input columns ส่วน target 3 ตัวถูกเก็บแยกเป็นอีก DataFrame จากนั้นบังคับลำดับคอลัมน์ของ test ให้ตรงกับ train และตรวจจำนวนแถวให้จับคู่กันได้

ตาราง shape report ควรเห็น feature 157 คอลัมน์และ target 3 คอลัมน์ในทุกชุด หาก `feature_schema_matches_train` เป็น True และจำนวนแถว feature เท่ากับ target แปลว่ารูปแบบ input สอดคล้องกัน แต่ยังต้องผ่านการตรวจ missing/infinite ในขั้นถัดไป

ไฟล์ที่สร้างมีทั้ง full feature set 157 คอลัมน์และ target แยกต่างหาก สำหรับ Isolation Forest baseline ในโปรเจกต์นี้ สคริปต์เลือกเฉพาะ original numeric + One-Hot = 121 คอลัมน์ จึงไม่ป้อน 36 engineered features เข้าโมเดล จนกว่าจะทดลองเปรียบเทียบเป็นชุด ๆ

In [8]:
def make_feature_matrix(frame):
    original_numeric = frame[numeric_cols].copy()
    engineered = build_engineered_columns(frame)
    categorical_encoded = encode_categories(frame).reindex(
        columns=ohe_columns,
        fill_value=0,
    )

    return pd.concat(
        [original_numeric, engineered, categorical_encoded],
        axis=1,
    )

feature_datasets = {}
target_datasets = {}

for dataset_name, frame in datasets.items():
    feature_datasets[dataset_name] = make_feature_matrix(frame)
    target_datasets[dataset_name] = frame[target_cols].copy()

train_feature_columns = feature_datasets["KDDTrain"].columns.tolist()
for dataset_name in feature_datasets:
    feature_datasets[dataset_name] = feature_datasets[dataset_name].reindex(
        columns=train_feature_columns,
        fill_value=0,
    )

shape_report = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(feature_datasets[name]),
        "numeric_and_engineered_features": len(numeric_cols) + engineered_preview.shape[1],
        "one_hot_features": len(ohe_columns),
        "total_model_features": feature_datasets[name].shape[1],
        "target_columns": target_datasets[name].shape[1],
    }
    for name in feature_datasets
])
display(shape_report)

assert all(
    list(frame.columns) == train_feature_columns
    for frame in feature_datasets.values()
), "Train/test feature columns do not match"

assert all(
    len(feature_datasets[name]) == len(target_datasets[name])
    for name in feature_datasets
), "Features and targets row counts do not match"

,dataset,rows,numeric_and_engineered_features,one_hot_features,total_model_features,target_columns
0,KDDTrain,125973,73,84,157,3
1,KDDTest,22544,73,84,157,3
2,KDDTest-21,11850,73,84,157,3


## 8. ดูตัวอย่างค่าก่อนและหลังแปลง

ตารางตัวอย่างวางคอลัมน์เดิม (`src_bytes`, `dst_bytes`, count และ error rates) ไว้ข้างคอลัมน์ใหม่ที่คำนวณจากแถวเดียวกัน เช่น total bytes, log bytes, byte difference และ indicator ตัวอย่างนี้ช่วยตรวจสูตรด้วยสายตา: เมื่อไม่มี failed login ค่า `has_num_failed_logins` ต้องเป็น 0; เมื่อ bytes ฝั่ง source มากกว่า destination ค่า signed difference ควรเป็นบวก

แถวตัวอย่างใช้เพื่ออธิบายสูตร ไม่ใช่สถิติแทนทั้งชุด และไม่มี target ถูกนำมาคำนวณ feature

In [9]:
example = pd.concat(
    [
        datasets["KDDTrain"][[
            "src_bytes", "dst_bytes", "count", "srv_count",
            "serror_rate", "srv_serror_rate", "num_failed_logins",
        ]].head(5),
        engineered_preview[[
            "total_bytes", "log1p_src_bytes",
            "signed_log_byte_difference", "overall_error_rate",
            "syn_error_mean",
            "has_num_failed_logins", "has_root_activity",
        ]].head(5),
    ],
    axis=1,
)
display(example)

,src_bytes,dst_bytes,count,srv_count,serror_rate,srv_serror_rate,num_failed_logins,total_bytes,log1p_src_bytes,signed_log_byte_difference,overall_error_rate,syn_error_mean,has_num_failed_logins,has_root_activity
0,491,0,2,2,0.0000,0.0000,0,491,6.1985,6.1985,0.0000,0.0000,0,0
1,146,0,13,1,0.0000,0.0000,0,146,4.9904,4.9904,0.0000,0.0000,0,0
2,0,0,123,6,1.0000,1.0000,0,0,0.0000,0.0000,0.5000,1.0000,0,0
3,232,8153,5,5,0.2000,0.2000,0,8385,5.4510,-8.9774,0.1000,0.2000,0,0
4,199,420,30,32,0.0000,0.0000,0,619,5.2983,-5.4027,0.0000,0.0000,0,0


## 9. ตรวจความสมบูรณ์ของ features และ targets

validation report ตรวจจำนวนแถว feature/target, จำนวนคอลัมน์, missing และ infinite ใน features, missing ใน targets และการตรงกันของ schema ทุก split หากเงื่อนไขใดไม่ผ่านจะ raise error ก่อนเขียนผลลัพธ์

ผลที่ต้องการคือ feature rows เท่ากับ target rows, 157 feature columns, missing/infinite เป็น 0 และ schema match เป็น True ทั้งสามชุด ความหมายคือไฟล์มีรูปแบบพร้อมส่งให้โมเดลและไม่ทำให้แถว/คอลัมน์เหลื่อมกัน ไม่ได้ยืนยันว่า feature ทุกตัวมีประโยชน์หรือโมเดลจะตรวจจับได้แม่น

In [10]:
validation_rows = []

for dataset_name, feature_matrix in feature_datasets.items():
    target_frame = target_datasets[dataset_name]
    values = feature_matrix.to_numpy(dtype=float)

    validation_rows.append({
        "dataset": dataset_name,
        "feature_rows": len(feature_matrix),
        "target_rows": len(target_frame),
        "feature_columns": feature_matrix.shape[1],
        "missing_features": int(feature_matrix.isna().sum().sum()),
        "infinite_features": int(np.isinf(values).sum()),
        "target_missing": int(target_frame.isna().sum().sum()),
        "feature_schema_matches_train": (
            list(feature_matrix.columns) == train_feature_columns
        ),
    })

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

if (
    validation_report["missing_features"].sum() > 0
    or validation_report["infinite_features"].sum() > 0
    or validation_report["target_missing"].sum() > 0
    or not validation_report["feature_schema_matches_train"].all()
):
    raise ValueError("Feature validation failed; inspect validation_report.")

,dataset,feature_rows,target_rows,feature_columns,missing_features,infinite_features,target_missing,feature_schema_matches_train
0,KDDTrain,125973,125973,157,0,0,0,True
1,KDDTest,22544,22544,157,0,0,0,True
2,KDDTest-21,11850,11850,157,0,0,0,True


## 10. ส่งออก CSV และ metadata สำหรับตรวจย้อนกลับ

บันทึก `*_features.csv` กับ `*_targets.csv` แยกกันทุก split พร้อม `feature_schema.csv` ที่บอกชนิดและคำอธิบายของแต่ละคอลัมน์, `unknown_category_report.csv`, `feature_validation_report.csv` และ `feature_engineering_config.json` ที่ระบุค่ากำหนด/นโยบายที่ใช้

เมื่อเปิดรายงาน unknown category ให้ดูจำนวนหมวดและจำนวนแถวที่ได้รับผล; รอบปัจจุบันคาดว่าเป็นศูนย์ เพราะทุกหมวด test อยู่ใน vocabulary train หากไม่ใช่ศูนย์ไม่ได้แปลว่า pipeline พังทันที แต่ต้องเข้าใจว่าโมเดลจะเห็นเวกเตอร์ศูนย์ในกลุ่มหมวดนั้นและอาจต้องมีนโยบาย unknown category เพิ่มเติม

In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for dataset_name in feature_datasets:
    feature_path = OUTPUT_DIR / f"{dataset_name}_features.csv"
    target_path = OUTPUT_DIR / f"{dataset_name}_targets.csv"

    feature_datasets[dataset_name].to_csv(feature_path, index=False)
    target_datasets[dataset_name].to_csv(target_path, index=False)

    print(f"Saved features: {feature_path.relative_to(PROJECT_ROOT)}")
    print(f"Saved targets:  {target_path.relative_to(PROJECT_ROOT)}")

feature_schema = pd.DataFrame({
    "feature": train_feature_columns,
    "kind": [
        (
            "one_hot"
            if col in ohe_columns
            else "engineered"
            if col in engineered_preview.columns
            else "original_numeric"
        )
        for col in train_feature_columns
    ],
})
feature_schema["description"] = feature_schema["feature"].map(
    lambda col: (
        engineered_descriptions.get(col, f"One-Hot category: {col}")
        if col in engineered_preview.columns or col in ohe_columns
        else f"Numeric feature from cleaned dataset: {col}"
    )
)
feature_schema.to_csv(OUTPUT_DIR / "feature_schema.csv", index=False)

unknown_category_report.to_csv(
    OUTPUT_DIR / "unknown_category_report.csv",
    index=False,
)
validation_report.to_csv(
    OUTPUT_DIR / "feature_validation_report.csv",
    index=False,
)

config = {
    "input_directory": str(CLEAN_DIR.resolve()),
    "output_directory": str(OUTPUT_DIR.resolve()),
    "categorical_columns": categorical_cols,
    "target_columns_separate_file": target_cols,
    "numeric_columns_from_cleaned_data": numeric_cols,
    "log1p_columns": log_cols,
    "indicator_columns": indicator_cols,
    "one_hot_categories_fitted_from": "KDDTrain only",
    "unseen_test_category_policy": "all-zero one-hot vector; reported in unknown_category_report.csv",
    "scaling": "not applied; fit a scaler on training data inside the model pipeline when needed",
}
with (OUTPUT_DIR / "feature_engineering_config.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(config, file, indent=2, ensure_ascii=False)

print("\nOutput files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

Saved features: data-pipeline\output\NSL-KDD\feature_engineered\KDDTrain_features.csv
Saved targets:  data-pipeline\output\NSL-KDD\feature_engineered\KDDTrain_targets.csv


Saved features: data-pipeline\output\NSL-KDD\feature_engineered\KDDTest_features.csv
Saved targets:  data-pipeline\output\NSL-KDD\feature_engineered\KDDTest_targets.csv


Saved features: data-pipeline\output\NSL-KDD\feature_engineered\KDDTest-21_features.csv
Saved targets:  data-pipeline\output\NSL-KDD\feature_engineered\KDDTest-21_targets.csv

Output files:
- feature_engineering_config.json: 2,137 bytes
- feature_schema.csv: 10,679 bytes
- feature_validation_report.csv: 243 bytes
- KDDTest-21_features.csv: 6,107,529 bytes
- KDDTest-21_targets.csv: 208,572 bytes
- KDDTest_features.csv: 11,552,858 bytes
- KDDTest_targets.csv: 384,048 bytes
- KDDTrain_features.csv: 63,473,995 bytes
- KDDTrain_targets.csv: 2,037,744 bytes
- unknown_category_report.csv: 301 bytes


## 11. วิธีนำไฟล์ไปเทรนและข้อควรระวังเรื่อง target leakage

ไฟล์ `*_features.csv` คือ input; `*_targets.csv` คือคำตอบอ้างอิง ตัวอย่าง supervised multiclass ใช้ `attack_family`, binary ใช้ `is_attack` ส่วน `label` ใช้เมื่ออยากจำแนกชื่อ attack รายตัว ห้าม concat targets กลับเข้า X โดยไม่ตั้งใจ เพราะจะทำให้ผลดูดีจากการรั่วไหลของคำตอบ

สำหรับ Isolation Forest แบบ unsupervised ให้ fit ด้วย X เท่านั้นและเก็บ targets ไว้ใช้หลัง scoring เพื่อประเมินผลหรือวิเคราะห์ cluster; ไม่ใช้ label เลือก threshold บน KDDTest หากต้องปรับ threshold ให้ใช้ validation จาก train แล้วกัน KDDTest ไว้เป็นผลสุดท้าย

ชุด full feature มี 157 คอลัมน์ แต่สคริปต์ IF ปัจจุบันใช้ baseline 121 คอลัมน์ตาม schema (`original_numeric` + `one_hot`) เพื่อแยกผลของ feature engineering ออกจาก baseline การทดลองต่อไปสามารถเพิ่ม engineered features ทีละกลุ่มแล้วเทียบ metric ภายใต้ split/threshold เดียวกัน